# Graph-TCN-VAE: Nanzi demo workflow

This notebook runs the public package on a compact 96-hour excerpt from the Nanzi aerosol supersite record. It demonstrates the interface only: the excerpt is too small for scientifically valid reconstruction or calibrated uncertainty.

Workflow: inspect data → validate the hourly grid → train a small model → save/load a bundle → impute natural gaps → inspect uncertainty and support-risk fields.

In [ ]:
from pathlib import Path
import pandas as pd

from graph_tcn_vae import TrainConfig, impute, load_bundle, train_from_config
from graph_tcn_vae.data import load_frame

ROOT = Path.cwd()
if not (ROOT / 'examples').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'examples' / 'data' / 'nanzi_demo_96h.csv'
OUTPUT_DIR = ROOT / 'outputs' / 'nanzi_demo'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE = OUTPUT_DIR / 'nanzi_demo_bundle.pt'
IMPUTED = OUTPUT_DIR / 'nanzi_demo_imputed.csv'
DATA

In [ ]:
raw = pd.read_csv(DATA, parse_dates=['time'])
display(raw.head())
display(raw.isna().mean().sort_values(ascending=False).rename('missing_fraction').to_frame())

## Define the data contract

The first four targets are composition variables. The remaining four targets are representative PSD bins and are treated as a synchronized PSD family by setting `n_chem=4`. Auxiliary columns condition the reconstruction but are not imputed targets.

In [ ]:
TARGET_COLS = [
    'SO2', 'NO2', 'PM2.5', 'PM10',
    '11.8', '100.31334240877506', '491.3882093149086', '19810.0',
]
AUX_COLS = ['AT', 'RH', 'BLH', 'hour_sin', 'hour_cos']

validated = load_frame(
    [DATA],
    timestamp_col='time',
    target_cols=TARGET_COLS,
    aux_cols=AUX_COLS,
    expected_frequency='1h',
    time_grid_policy='strict',
)
print({
    'rows': len(validated),
    'start': validated.index.min(),
    'end': validated.index.max(),
    'frequency': validated.attrs['frequency'],
})

## Train a deliberately small demonstration model

The configuration below is designed to finish quickly on CPU/MPS. It is **not** the 26e research configuration. The demo keeps targets on their linear input scale to avoid unstable exponential back-transforms from a deliberately under-trained model; the audited 26e workflow uses log1p targets. For a scientific run, use substantially more data, the 26e parity settings, held-out evaluation, and multiple mask seeds.

In [ ]:
config = TrainConfig(
    csv=[str(DATA)],
    timestamp_col='time',
    target_cols=TARGET_COLS,
    aux_cols=AUX_COLS,
    target_transform='none',
    target_output_transform='none',
    expected_frequency='1h',
    time_grid_policy='strict',
    window_size=24,
    stride=12,
    val_fraction=0.25,
    batch_size=4,
    epochs=3,
    patience=3,
    validation_metric='ho_mse',
    dynamic_mask_mean_duration=6,
    dynamic_mask_std_duration=2,
    dynamic_mask_min_duration=2,
    dynamic_mask_max_duration=12,
    model_kwargs={
        'n_chem': 4,
        'latent_dim': 16,
        'hidden_dims': [32, 32],
        'encoder_layers': 2,
        'decoder_layers': 2,
        'n_graph_heads': 1,
        'dropout': 0.0,
        'heteroscedastic': True,
    },
)
best_value = train_from_config(config, str(BUNDLE))
print('bundle:', BUNDLE)
print('best held-out selection value:', best_value)

## Load the self-contained bundle and impute

The bundle contains weights, model construction arguments, feature order, target/auxiliary scalers, transforms, stride, and the timestamp-grid contract.

In [ ]:
bundle = load_bundle(BUNDLE)
print({key: bundle[key] for key in [
    'target_cols', 'aux_cols', 'window_size', 'stride',
    'target_transform', 'target_output_transform', 'time_grid',
]})

result = impute(
    [DATA],
    bundle,
    IMPUTED,
    n_mc_samples=10,
    inference_batch_size=4,
    mc_batch_size=2,
)
print('output:', IMPUTED)
display(result.head())

## Inspect only imputed points

`heuristic_risk_tier` is an operational descriptor based on gap duration and bilateral observed context. It is not a calibrated probability and does not replace held-out evaluation.

In [ ]:
missing_predictions = result[result['is_imputed']].copy()
display(missing_predictions[[
    'timestamp', 'feature', 'imputed_mean', 'imputed_std', 'q05', 'q95',
    'gap_length', 'left_context_fraction', 'right_context_fraction',
    'heuristic_risk_tier',
]].head(20))
display(
    missing_predictions.groupby(['feature', 'heuristic_risk_tier'])
    .size()
    .rename('points')
    .to_frame()
)

## What to do next

For a real dataset, run `graph-tcn-vae validate-data` first, define feature families and transforms explicitly, train on a sufficiently long period, evaluate on fixed held-out gaps, and inspect performance separately by feature family, gap duration, and context support. Do not treat imputed values as direct measurements.